#### LangChain Utilities for RAG

- Using the Vector Store (vector_db) we have created earlier

In [1]:
''' 
%pip install -U langchain-groq
%pip install -U langchain-openai
'''

' \n%pip install -U langchain-groq\n%pip install -U langchain-openai\n'

In [14]:
# imports

import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
from langchain_openai import ChatOpenAI
from IPython.display import display, Markdown

In [3]:
# Model, DB

MODEL = 'openai/gpt-oss-20b'
DB_PATH = '../vector_db'

# loading and validating Groq API Key

load_dotenv(override=True)

groq_base_url = os.getenv('GROQ_BASE_URL')
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f'Groq API Key found and starts with {groq_api_key[0:3]}')
else:
    print('Groq API Key not found')


Groq API Key found and starts with gsk


#### Connecting to Chroma and use HuggingFaceEmbeddings to convert the question into vectors before retrieving from the Data Store

In [4]:
# setting up embedding model and connecting to vector data store

embeddings = HuggingFaceEmbeddings(model_name = 'all-MiniLM-L6-v2')
try:
    vector_store = Chroma(persist_directory=DB_PATH, embedding_function=embeddings)
    print(f'Successfully Connected to the vector store')
except Exception as e:
    print(f'Error occured while connecting to the Data Store : {e}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully Connected to the vector store


#### Temperature

**A sidebar on "temperature"**:

- Controls how diverse the output is

- A temperature of 0 means that the output should be predictable

- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right

- It actually controls which tokens get selected during inference

- temperature=0 means: always select the token with highest probability

- temperature=1 usually means: a token with 10% probability should be picked 10% of the time - probability distribution

**Note-1**: 

- A temperature of 0 doesn't mean outputs will always be reproducible. 

- You also need to set a random seed. (Even then, it's not always reproducible.)

**Note-2**: 

- If you want creativity, use the System Prompt!

#### Setting up 2 LangChain objects

1. retriever

2. llm

In [5]:
# retriever and llm

retriever = vector_store.as_retriever()
llm = ChatOpenAI(temperature=0, model=MODEL, base_url=groq_base_url, api_key=groq_api_key)

In [6]:
# implementing the LangChain object 'retriever' using invoke() method

retriever.invoke('Who is Avery?')

[Document(id='f461226c-5af0-469b-9f4e-878a183a8e97', metadata={'source': '../knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and 

In [7]:
# implementing the LangChain object 'llm' using invoke() method

llm.invoke('Who is Avery?')

AIMessage(content='I’m not sure which Avery you’re referring to—there are many people, characters, and even places named Avery. Could you let me know a bit more about the context? For example, are you asking about a public figure, a fictional character, a historical person, or something else entirely? That’ll help me give you the most accurate answer.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 133, 'prompt_tokens': 75, 'total_tokens': 208, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 53, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.282124327, 'prompt_time': 0.003541065, 'completion_time': 0.135808641, 'total_time': 0.139349706}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_37c9245f64', 'id': 'chatcmpl-39630998-7201-438f-bf65-080081b360f7', 'service_tier': 'on_demand', '

#### Retriever and LLM Together

In [8]:
# system prompt prefix

SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
# RAG

def answer_question(question: str, history):
    docs = retriever.invoke(question)                                                               # retrieve the relevant chunks from the vector store using retriever
    context = '\n\n'.join(doc.page_content for doc in docs)                                         # join all the relevant chunks
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)                                  # format the relevant chunks into system message
    response = llm.invoke([SystemMessage(content=system_prompt),HumanMessage(content=question)])    # invoke the llm with latest system message (along with relevant context) and question
    return response.content                                                                         # return response


In [ ]:
# Invoking RAG

display(Markdown(answer_question('Who is Avery?', [])))             # since the function requires 2 arguments and we are not handling the history yet, so [] is passed for temporary

Avery Lancaster is the Co‑Founder and Chief Executive Officer (CEO) of **Insurellm**, a leading insurance‑technology company based in San Francisco, California.  

**Key highlights about Avery:**

| Detail | Information |
|--------|-------------|
| **Full name** | Avery Lancaster |
| **Date of birth** | March 15, 1985 |
| **Current role** | Co‑Founder & CEO of Insurellm |
| **Location** | San Francisco, California |
| **Current salary** | $225,000 |
| **Career at Insurellm** | 2015‑present – has steered the company from its founding to a market‑leading position in insurance tech |
| **Prior experience** | 2013‑2015 – Senior Product Manager at Innovate Insurance Solutions, where she developed cutting‑edge products for the tech sector |
| **Leadership style** | Known for innovative strategies, risk‑management expertise, and a strong focus on diversity, inclusion, and community outreach |
| **Performance record** | Consistently exceeded expectations in recent years, with notable achievements in 2021 (remote‑work transition), 2023 (market leadership), and 2018 (product launches) |
| **Professional development** | Actively participates in leadership training, industry conferences, and fosters partnerships |
| **Community engagement** | Leads financial‑literacy programs for underserved populations, boosting Insurellm’s CSR profile |

Avery’s blend of entrepreneurial vision, product innovation, and people‑centric leadership has positioned Insurellm as a key player in the insurance technology landscape. If you’d like to know more about her initiatives or Insurellm’s offerings, just let me know!

In [16]:
# gradio UI

gr.ChatInterface(fn=answer_question).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
